## SelfGNN — Yelp Merchant Dataset Preprocessing (Corrected)

Fixes applied:
1. Category filter now actually filters (empty string excluded)
2. Test/val split removes only the held-out timestamp, not all visits
3. Test and val get independent negative samples
4. Sequence preserves every individual visit (not collapsed per merchant)
5. No temporal leak: test target NOT appended to sequence
6. Post-deletion minimum interaction check
7. val_dict saved as separate pickle
8. trans() counts every visit event, matching sequence-based matrix

In [1]:
OUT_DIR = './Datasets/yelp-merchant/'
RAW_DIR = './Datasets/yelp_dataset/'

import os
os.makedirs(OUT_DIR, exist_ok=True)

import json
import pickle
import copy
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import scipy.sparse as sp

print('Libraries loaded. Output dir:', OUT_DIR)

Libraries loaded. Output dir: ./Datasets/yelp-merchant/


In [2]:
merchant_bids = set()
bid_to_name = {}

business_path = RAW_DIR + 'yelp_academic_dataset_business.json'
with open(business_path, 'r', encoding='utf-8') as f:
    for line in f:
        b = json.loads(line)
        cats = b.get('categories') or ''
        # FIX: filter out empty tokens so '' doesn't pass as a category
        cat_tokens = {c.strip() for c in cats.split(',') if c.strip()}
        if cat_tokens:
            bid = b['business_id']
            merchant_bids.add(bid)
            bid_to_name[bid] = b.get('name', '')

print(f'Merchant businesses found: {len(merchant_bids):,}')
print('Example names:', list(bid_to_name.values())[:5])

Merchant businesses found: 150,243
Example names: ['Abby Rappoport, LAC, CMQ', 'The UPS Store', 'Target', 'St Honore Pastries', 'Perkiomen Valley Brewery']


In [3]:
def parse_date(date_str):
    """Parse Yelp date string 'YYYY-MM-DD HH:MM:SS' -> Unix timestamp (int)."""
    try:
        return int(datetime.strptime(date_str, '%Y-%m-%d %H:%M:%S').timestamp())
    except Exception:
        return None

review_path = RAW_DIR + 'yelp_academic_dataset_review.json'

raw_interactions = defaultdict(lambda: defaultdict(list))

n_read = 0
n_kept = 0

with open(review_path, 'r', encoding='utf-8') as f:
    for line in f:
        n_read += 1
        rv = json.loads(line)
        bid = rv.get('business_id', '')
        if bid not in merchant_bids:
            continue
        uid = rv.get('user_id', '')
        ts = parse_date(rv.get('date', ''))
        if uid and bid and ts:
            raw_interactions[uid][bid].append(ts)
            n_kept += 1
        if n_read % 1_000_000 == 0:
            print(f'  Read {n_read:,} reviews, kept {n_kept:,}')

print(f'\nTotal reviews read : {n_read:,}')
print(f'Merchant reviews   : {n_kept:,}')
print(f'Unique users       : {len(raw_interactions):,}')
print(f'Unique merchants   : {len({b for u in raw_interactions.values() for b in u}):,}')

  Read 1,000,000 reviews, kept 999,934
  Read 2,000,000 reviews, kept 1,999,808
  Read 3,000,000 reviews, kept 2,999,710
  Read 4,000,000 reviews, kept 3,999,622
  Read 5,000,000 reviews, kept 4,999,524
  Read 6,000,000 reviews, kept 5,999,455

Total reviews read : 6,990,280
Merchant reviews   : 6,989,591
Unique users       : 1,987,685
Unique merchants   : 150,243


In [4]:
MIN_INTERACTIONS = 5

def kcore_filter(interactions, min_u=MIN_INTERACTIONS, min_i=MIN_INTERACTIONS):
    data = {u: dict(merchants) for u, merchants in interactions.items()}
    iteration = 0
    while True:
        merchant_cnt = defaultdict(int)
        for u, merchants in data.items():
            for i in merchants:
                merchant_cnt[i] += 1
        valid_merchants = {i for i, cnt in merchant_cnt.items() if cnt >= min_i}

        data = {u: {i: ts for i, ts in merchants.items() if i in valid_merchants}
                for u, merchants in data.items()}
        data = {u: merchants for u, merchants in data.items() if len(merchants) >= min_u}

        merchant_cnt2 = defaultdict(int)
        for u, merchants in data.items():
            for i in merchants:
                merchant_cnt2[i] += 1
        still_valid = {i for i, cnt in merchant_cnt2.items() if cnt >= min_i}

        iteration += 1
        n_users = len(data)
        n_merchants = len(still_valid)
        n_ints = sum(len(v) for v in data.values())
        print(f'  Iter {iteration}: {n_users:,} users, {n_merchants:,} merchants, {n_ints:,} interactions')

        if still_valid == valid_merchants and all(len(m) >= min_u for m in data.values()):
            break
        if n_users == 0 or n_merchants == 0:
            break
    return data

print('K-core filtering (min=5)...')
filtered = kcore_filter(raw_interactions)
print(f'After k-core: {len(filtered):,} users')
all_merchants_kcore = {i for u in filtered.values() for i in u}
print(f'After k-core: {len(all_merchants_kcore):,} merchants')

K-core filtering (min=5)...
  Iter 1: 279,089 users, 110,587 merchants, 4,153,042 interactions
  Iter 2: 269,148 users, 109,374 merchants, 4,006,537 interactions
  Iter 3: 268,668 users, 109,327 merchants, 3,999,831 interactions
  Iter 4: 268,653 users, 109,325 merchants, 3,999,583 interactions
  Iter 5: 268,653 users, 109,325 merchants, 3,999,575 interactions
After k-core: 268,653 users
After k-core: 109,325 merchants


In [5]:
user_strs = sorted(filtered.keys())
merchant_strs = sorted(all_merchants_kcore)

user2id = {u: i for i, u in enumerate(user_strs)}
merchant2id = {it: i for i, it in enumerate(merchant_strs)}

usrnum = len(user_strs)
merchantnum = len(merchant_strs)

interaction = [None] * usrnum
minn = int(2e12)
maxx = 0

for u_str, merchants in filtered.items():
    u = user2id[u_str]
    interaction[u] = {}
    for i_str, tss in merchants.items():
        if i_str not in merchant2id:
            continue
        it = merchant2id[i_str]
        interaction[u][it] = sorted(tss)
        for ts in tss:
            if ts < minn: minn = ts
            if ts > maxx: maxx = ts

print(f'usrnum = {usrnum:,}')
print(f'merchantnum = {merchantnum:,}')
print(f'Time range: {datetime.fromtimestamp(minn)} -> {datetime.fromtimestamp(maxx)}')

n_total_interactions = sum(
    sum(len(tss) for tss in interaction[u].values())
    for u in range(usrnum) if interaction[u] is not None
)
n_total_pairs = sum(len(v) for v in interaction if v is not None)
density = n_total_pairs / (usrnum * merchantnum) * 100
print(f'Total interaction events: {n_total_interactions:,}')
print(f'Unique (user, merchant) pairs: {n_total_pairs:,}')
print(f'Density: {density:.4f}%')

usrnum = 268,653
merchantnum = 109,325
Time range: 2005-02-16 03:23:22 -> 2022-01-19 19:48:45
Total interaction events: 4,190,201
Unique (user, merchant) pairs: 3,999,575
Density: 0.0136%


In [6]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    user_degrees = [len(interaction[u]) for u in range(usrnum) if interaction[u] is not None]
    merchant_deg = defaultdict(int)
    for u in range(usrnum):
        if interaction[u]:
            for it in interaction[u]:
                merchant_deg[it] += 1
    merchant_degrees_list = list(merchant_deg.values())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Yelp-Merchant Dataset Statistics', fontweight='bold')
    axes[0].hist(user_degrees, bins=50, color='#1976D2', alpha=0.8)
    axes[0].set_xlabel('Merchants per user')
    axes[0].set_ylabel('Count')
    axes[0].set_title('User degree distribution')
    axes[0].set_yscale('log')
    axes[1].hist(merchant_degrees_list, bins=50, color='#F57C00', alpha=0.8)
    axes[1].set_xlabel('Users per merchant')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Merchant degree distribution')
    axes[1].set_yscale('log')
    plt.tight_layout()
    plt.savefig(OUT_DIR + 'dataset_stats.png', bbox_inches='tight', dpi=120)
    plt.show()
except ImportError:
    print('matplotlib not available, skipping plot')

print(f'Users:              {usrnum:>10,}')
print(f'Merchants:          {merchantnum:>10,}')
print(f'Unique pairs:       {n_total_pairs:>10,}')
print(f'Total events:       {n_total_interactions:>10,}')
print(f'Density:            {density:>10.4f}%')

Users:                 268,653
Merchants:             109,325
Unique pairs:        3,999,575
Total events:        4,190,201
Density:                0.0136%


/var/folders/sr/2htv7_wj6sj6qtmjtk9f1vpc0000gn/T/ipykernel_52454/3415173836.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
def negSamp(pos_set, sampSize, nodeNum):
    """Sample `sampSize` negative merchants for a user.
    pos_set: set of merchant IDs already interacted with.
    """
    valid = [j for j in range(nodeNum) if j not in pos_set]
    if len(valid) == 0:
        valid = list(range(nodeNum))
    sampSize = min(sampSize, len(valid))
    idx = np.random.choice(len(valid), size=sampSize, replace=False)
    return [valid[i] for i in idx]

In [8]:
def split_test_val(interaction, usrnum, merchantnum):
    """
    Chronological split:
      - Most recent interaction -> test
      - Second most recent (different merchant) -> val
      - Everything else -> train

    Fixes:
      1. Only removes the specific held-out timestamp, keeps earlier visits
      2. Independent negative samples for test and val
      3. Post-deletion minimum check (need >= 1 training interaction)
    """
    np.random.seed(100)
    pickNum = min(10000, usrnum)

    eligible = [u for u in range(usrnum)
                if interaction[u] is not None and len(interaction[u]) >= 3]
    perm = np.random.permutation(eligible)
    pickUsr = perm[:pickNum]

    tstInt = [None] * usrnum
    valInt = [None] * usrnum
    test_rows = []
    val_rows = []
    skipped = 0

    for u in pickUsr:
        merchants = interaction[u]
        if merchants is None or len(merchants) < 3:
            skipped += 1
            continue

        pairs = []
        for it, tss in merchants.items():
            for ts in tss:
                pairs.append((ts, it))
        pairs.sort()

        tst_time, tst_merchant = pairs[-1]

        val_merchant = None
        val_time = None
        for ts, it in reversed(pairs[:-1]):
            if it != tst_merchant:
                val_merchant = it
                val_time = ts
                break

        if val_merchant is None:
            skipped += 1
            continue

        # FIX: Remove only the specific timestamp, not the entire merchant
        tss_tst = interaction[u][tst_merchant]
        tss_tst = [t for t in tss_tst if t != tst_time]
        if len(tss_tst) == 0:
            del interaction[u][tst_merchant]
        else:
            interaction[u][tst_merchant] = tss_tst

        tss_val = interaction[u].get(val_merchant, [])
        tss_val = [t for t in tss_val if t != val_time]
        if len(tss_val) == 0:
            if val_merchant in interaction[u]:
                del interaction[u][val_merchant]
        else:
            interaction[u][val_merchant] = tss_val

        # FIX: Post-deletion check
        remaining = {m for m, tss in interaction[u].items() if tss}
        if len(remaining) < 1:
            skipped += 1
            continue

        tstInt[u] = tst_merchant
        valInt[u] = val_merchant

        all_pos = set(merchants.keys())

        # FIX: Independent negatives
        test_negs = negSamp(all_pos, 999, merchantnum)
        val_negs = negSamp(all_pos, 999, merchantnum)

        test_rows.append({
            'user_id': u + 1,
            'merchant_id': tst_merchant,
            'time': tst_time,
            'neg_merchants': str(test_negs),
        })
        val_rows.append({
            'user_id': u + 1,
            'merchant_id': val_merchant,
            'time': val_time,
            'neg_merchants': str(val_negs),
        })

    pd.DataFrame(test_rows).to_csv(OUT_DIR + 'test_yelp_merchant.csv', sep='\t', index=False)
    pd.DataFrame(val_rows).to_csv(OUT_DIR + 'val_yelp_merchant.csv', sep='\t', index=False)

    n_test = sum(1 for x in tstInt if x is not None)
    n_val = sum(1 for x in valInt if x is not None)
    print(f'Test users:  {n_test:,}')
    print(f'Val  users:  {n_val:,}')
    print(f'Skipped:     {skipped:,}')
    return interaction, tstInt, valInt


interaction_train = copy.deepcopy(interaction)
interaction_train, tstInt, valInt = split_test_val(interaction_train, usrnum, merchantnum)

Test users:  10,000
Val  users:  10,000
Skipped:     0


In [9]:
def trans(interaction, usrnum, merchantnum):
    """Build global user-merchant CSR matrix from training interactions.
    Each (user, merchant, timestamp) event adds 1 so repeated visits
    are reflected, matching the sequence-based matrix in data_handler.
    """
    r, c, d = [], [], []
    for usr in range(usrnum):
        if interaction[usr] is None:
            continue
        for col, tss in interaction[usr].items():
            if not tss:
                continue
            for _ in tss:
                r.append(usr)
                c.append(col)
                d.append(1.0)
    mat = csr_matrix((d, (r, c)), shape=(usrnum, merchantnum))
    binary = (mat != 0).astype(np.float32)
    print(f'Global matrix: shape {mat.shape}, '
          f'{binary.nnz:,} unique pairs, '
          f'{len(d):,} total events')
    return mat

In [10]:
GRAPH_NUM = 5

def trans_sub(interaction, usrnum, merchantnum, graph_num):
    """Build T binary time-interval sparse matrices + timeMat."""
    global minn, maxx
    interval = (maxx - minn) / graph_num

    rcd = [[[], [], []] for _ in range(graph_num)]
    seen = [set() for _ in range(graph_num)]
    last_interval = {}

    for usr in range(usrnum):
        if interaction[usr] is None:
            continue
        for col, tss in interaction[usr].items():
            if not tss:
                continue
            for ts in tss:
                t = int((ts - minn) / interval)
                t = max(0, min(t, graph_num - 1))
                key = (usr, col)
                if key not in seen[t]:
                    rcd[t][0].append(usr)
                    rcd[t][1].append(col)
                    rcd[t][2].append(1.0)
                    seen[t].add(key)
                if key not in last_interval or last_interval[key] < t:
                    last_interval[key] = t

    intMat = []
    for t in range(graph_num):
        mat = csr_matrix(
            (rcd[t][2], (rcd[t][0], rcd[t][1])),
            shape=(usrnum, merchantnum))
        intMat.append(mat)
        print(f'  A_{t}: {mat.nnz:,} non-zeros')

    tmat_r, tmat_c, tmat_d = [], [], []
    for (u, i), t in last_interval.items():
        tmat_r.append(u)
        tmat_c.append(i)
        tmat_d.append(t)
    timeMat = csr_matrix((tmat_d, (tmat_r, tmat_c)), shape=(usrnum, merchantnum))
    print(f'  timeMat: {timeMat.nnz:,} non-zeros')
    return intMat, timeMat

print(f'Building T={GRAPH_NUM} binary time-interval matrices...')
subMat, timeMat = trans_sub(interaction_train, usrnum, merchantnum, GRAPH_NUM)

Building T=5 binary time-interval matrices...
  A_0: 32,561 non-zeros
  A_1: 330,450 non-zeros
  A_2: 914,624 non-zeros
  A_3: 1,540,756 non-zeros
  A_4: 1,213,684 non-zeros
  timeMat: 3,980,492 non-zeros


In [11]:
global_mat = trans(interaction_train, usrnum, merchantnum)

train_rows = []
for u in range(usrnum):
    if interaction_train[u] is None:
        continue
    for it, tss in interaction_train[u].items():
        if tss:
            for ts in tss:
                train_rows.append({'user_id': u + 1, 'merchant_id': it, 'time': ts})

df_train = pd.DataFrame(train_rows)
df_train.to_csv(OUT_DIR + 'train_yelp_merchant.csv', sep='\t', index=False)
print(f'Train CSV saved: {len(df_train):,} rows')

trnMat = [global_mat, subMat, timeMat]
with open(OUT_DIR + 'trn_mat_time', 'wb') as fs:
    pickle.dump(trnMat, fs)
print('Saved: trn_mat_time')

with open(OUT_DIR + 'tst_int', 'wb') as fs:
    pickle.dump(tstInt, fs)
with open(OUT_DIR + 'val_int', 'wb') as fs:
    pickle.dump(valInt, fs)
print('Saved: tst_int, val_int')

print(f'\ntrnMat shapes: global={global_mat.shape}, T={len(subMat)}, timeMat={timeMat.shape}')

Global matrix: shape (268653, 109325), 3,980,492 unique pairs, 4,170,201 total events
Train CSV saved: 4,170,201 rows
Saved: trn_mat_time
Saved: tst_int, val_int

trnMat shapes: global=(268653, 109325), T=5, timeMat=(268653, 109325)


In [12]:
import ast

# FIX: Every individual visit as a separate entry
# FIX: Do NOT append test target (avoids temporal leak in val)
sequence = []
empty_count = 0
for u in range(usrnum):
    if interaction_train[u] is None or len(interaction_train[u]) == 0:
        sequence.append([0])
        empty_count += 1
        continue

    pairs = []
    for it, tss in interaction_train[u].items():
        if tss:
            for ts in tss:
                pairs.append((ts, it))
    pairs.sort()

    if len(pairs) == 0:
        sequence.append([0])
        empty_count += 1
        continue

    seq = [it for _, it in pairs]
    sequence.append(seq)

with open(OUT_DIR + 'sequence', 'wb') as fs:
    pickle.dump(sequence, fs)

non_empty = usrnum - empty_count
print(f'Saved: sequence ({len(sequence):,} users, {non_empty:,} non-empty)')

# Verify: sequence event count should match global_mat total events
seq_events = sum(len(s) for s in sequence) - empty_count  # subtract placeholders
print(f'Sequence total events: {seq_events:,}')
print(f'Global matrix total events: {int(global_mat.sum()):,}')
assert seq_events == int(global_mat.sum()), \
    f'MISMATCH: sequence has {seq_events} events but global_mat has {int(global_mat.sum())}'
print('Sequence-matrix consistency check passed.')

# test_dict (0-indexed keys)
test_csv = pd.read_csv(OUT_DIR + 'test_yelp_merchant.csv', sep='\t')
test_dict = {}
for _, row in test_csv.iterrows():
    uid = int(row['user_id']) - 1
    negs = ast.literal_eval(str(row['neg_merchants']))
    test_dict[uid] = negs

with open(OUT_DIR + 'test_dict', 'wb') as fs:
    pickle.dump(test_dict, fs)
print(f'Saved: test_dict ({len(test_dict):,} users, 0-indexed keys)')

# val_dict (0-indexed keys)
val_csv = pd.read_csv(OUT_DIR + 'val_yelp_merchant.csv', sep='\t')
val_dict = {}
for _, row in val_csv.iterrows():
    uid = int(row['user_id']) - 1
    negs = ast.literal_eval(str(row['neg_merchants']))
    val_dict[uid] = negs

with open(OUT_DIR + 'val_dict', 'wb') as fs:
    pickle.dump(val_dict, fs)
print(f'Saved: val_dict ({len(val_dict):,} users, 0-indexed keys)')

Saved: sequence (268,653 users, 268,653 non-empty)
Sequence total events: 4,170,201
Global matrix total events: 4,170,201
Sequence-matrix consistency check passed.
Saved: test_dict (10,000 users, 0-indexed keys)
Saved: val_dict (10,000 users, 0-indexed keys)


In [13]:
df_train = pd.read_csv(OUT_DIR + 'train_yelp_merchant.csv', sep='\t')
df_test = pd.read_csv(OUT_DIR + 'test_yelp_merchant.csv', sep='\t')
df_val = pd.read_csv(OUT_DIR + 'val_yelp_merchant.csv', sep='\t')

train_pairs = set(zip(df_train['user_id'], df_train['merchant_id']))
test_pairs = set(zip(df_test['user_id'], df_test['merchant_id']))
val_pairs = set(zip(df_val['user_id'], df_val['merchant_id']))

overlap_tr_ts = train_pairs & test_pairs
overlap_tr_vl = train_pairs & val_pairs
overlap_ts_vl = test_pairs & val_pairs

print(f'Train-Test overlap:  {len(overlap_tr_ts)} pairs')
print(f'Train-Val  overlap:  {len(overlap_tr_vl)} pairs')
print(f'Test-Val   overlap:  {len(overlap_ts_vl)} pairs')

# NOTE: overlap_tr_ts and overlap_tr_vl may be > 0 when a user visited the
# same merchant multiple times and only the held-out timestamp was removed.
# This is CORRECT — the earlier visits are legitimate training data.
if len(overlap_tr_ts) > 0:
    print(f'  (Train-Test overlap is expected: earlier visits to the same merchant remain in train)')
if len(overlap_tr_vl) > 0:
    print(f'  (Train-Val overlap is expected: earlier visits to the same merchant remain in train)')

assert len(overlap_ts_vl) == 0, f'FAIL: Test-Val overlap: {len(overlap_ts_vl)} pairs!'
print('Test-Val disjointness check passed.')

# Verify no test target leak in sequences
leak_count = 0
for u in range(usrnum):
    if tstInt[u] is not None and tstInt[u] in sequence[u]:
        leak_count += 1
if leak_count > 0:
    print(f'Note: {leak_count} sequences contain test merchant '
          f'(legitimate: user visited that merchant earlier in training period)')
else:
    print('No test merchants found in training sequences.')

# Verify file structure
required_files = ['trn_mat_time', 'tst_int', 'val_int', 'sequence',
                  'test_dict', 'val_dict']
for fname in required_files:
    path = OUT_DIR + fname
    size = os.path.getsize(path) if os.path.isfile(path) else 0
    status = f'{size/1024:.0f} KB' if size > 0 else 'MISSING'
    print(f'  {fname:20s} {status}')

print()
print('=' * 60)
print('Preprocessing Complete')
print('=' * 60)
print(f'Dataset:             yelp-merchant')
print(f'Users:               {usrnum:>10,}')
print(f'Merchants:           {merchantnum:>10,}')
print(f'Train events:        {len(df_train):>10,}')
print(f'Test users:          {len(df_test):>10,}')
print(f'Val  users:          {len(df_val):>10,}')
print(f'Time intervals (T):  {GRAPH_NUM:>10}')
print(f'Output dir:          {OUT_DIR}')

Train-Test overlap:  494 pairs
Train-Val  overlap:  423 pairs
Test-Val   overlap:  0 pairs
  (Train-Test overlap is expected: earlier visits to the same merchant remain in train)
  (Train-Val overlap is expected: earlier visits to the same merchant remain in train)
Test-Val disjointness check passed.
Note: 494 sequences contain test merchant (legitimate: user visited that merchant earlier in training period)
  trn_mat_time         147891 KB
  tst_int              290 KB
  val_int              290 KB
  sequence             16488 KB
  test_dict            37144 KB
  val_dict             37152 KB

Preprocessing Complete
Dataset:             yelp-merchant
Users:                  268,653
Merchants:              109,325
Train events:         4,170,201
Test users:              10,000
Val  users:              10,000
Time intervals (T):           5
Output dir:          ./Datasets/yelp-merchant/
